# 05 Final Prediction, Spatial Analysis and Model Interpretation

This notebook converts the validated modelling outputs from notebook 003 into the final dissertation results.

The final mapping model is the **Tabular XGBoost** configuration because it achieved the strongest overall predictive performance in the nested cross-validation experiment. The CNN experiment remains part of the model-comparison evidence, but it is not used as the primary final mapping model.

Main tasks:

1. Load the final 003 nested-CV outputs and 500 m grid.
2. Summarise final model performance.
3. Plot observed versus OOF-predicted electricity consumption.
4. Create Random 5-fold OOF observed / predicted / residual maps.
5. Create Spatial K=4 OOF observed / predicted / residual maps.
6. Analyse spatial-block errors and generalisation.
7. Analyse prediction-error structure.
8. Fit one full-data explanatory XGBoost model using a hyperparameter configuration selected from inner-CV evidence and calculate feature importance / optional SHAP values.
9. Export final spatial prediction layers and result tables.

**Important:** OOF predictions are used for the validated prediction maps. A model trained on all labelled grids is used only for feature interpretation, not to replace OOF validation results.

In [ ]:
# ============================================================
# Imports and display settings
# ============================================================

import json
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.colors import LogNorm, SymLogNorm
import numpy as np
import pandas as pd
import seaborn as sns

from IPython.display import display

from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

from xgboost import XGBRegressor

sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
FINAL_FEATURE_SET = "Tabular"

print("Final mapping model:", FINAL_FEATURE_SET)

In [ ]:
# ============================================================
# Project paths
# ============================================================

import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from project_config import RAW_DATA_DIR, WORK_DIR

PROJECT_DATA_DIR = WORK_DIR

GRID_PATH = (
    PROJECT_DATA_DIR
    / "grid_size_selection"
    / "grids"
    / "grid_features_500m.gpkg"
)

FEATURE_MATRIX_PATH = (
    PROJECT_DATA_DIR
    / "grid_size_selection"
    / "features"
    / "feature_matrix_500m.csv"
)

SPATIAL_BLOCK_PATH = (
    PROJECT_DATA_DIR
    / "model_tuning_500m"
    / "spatial_block_assignments_500m.csv"
)

# Main 003 results
MODEL_RESULT_DIR = (
    PROJECT_DATA_DIR
    / "multimodal_fusion_500m"
)

OOF_PREDICTION_PATH = (
    MODEL_RESULT_DIR
    / "multimodal_oof_predictions.csv"
)

SUMMARY_PATH = (
    MODEL_RESULT_DIR
    / "multimodal_summary.csv"
)

FOLD_RESULT_PATH = (
    MODEL_RESULT_DIR
    / "multimodal_nested_fold_results.csv"
)

GAIN_PATH = (
    MODEL_RESULT_DIR
    / "cnn_incremental_gain.csv"
)


# Keep ResNet50 final-analysis outputs separate
OUTPUT_DIR = (
    PROJECT_DATA_DIR
    / "final_prediction_500m_resnet50_imagenet"
)

FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
SPATIAL_OUTPUT_DIR = OUTPUT_DIR / "spatial"

for directory in [
    OUTPUT_DIR,
    FIGURE_DIR,
    TABLE_DIR,
    SPATIAL_OUTPUT_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

required_paths = [
    GRID_PATH,
    FEATURE_MATRIX_PATH,
    SPATIAL_BLOCK_PATH,
    OOF_PREDICTION_PATH,
    SUMMARY_PATH,
    FOLD_RESULT_PATH,
]

for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(
            f"Required file not found: {path}"
        )

print("Grid:", GRID_PATH)
print("003 results:", MODEL_RESULT_DIR)
print("004 output:", OUTPUT_DIR)

In [ ]:
# ============================================================
# Load final 003 results and spatial data
# ============================================================

grid = (
    gpd.read_file(GRID_PATH)
    .sort_values("grid_id")
    .reset_index(drop=True)
)

feature_matrix = (
    pd.read_csv(FEATURE_MATRIX_PATH)
    .sort_values("grid_id")
    .reset_index(drop=True)
)

spatial_blocks = (
    pd.read_csv(SPATIAL_BLOCK_PATH)
    .sort_values("grid_id")
    .reset_index(drop=True)
)

oof_predictions = pd.read_csv(
    OOF_PREDICTION_PATH
)

model_summary = pd.read_csv(
    SUMMARY_PATH
)

fold_results = pd.read_csv(
    FOLD_RESULT_PATH
)

if GAIN_PATH.exists():
    cnn_gain = pd.read_csv(
        GAIN_PATH
    )
else:
    cnn_gain = pd.DataFrame()

required_prediction_columns = {
    "grid_id",
    "validation",
    "feature_set",
    "fold",
    "observed",
    "predicted",
    "residual",
}

if not required_prediction_columns.issubset(
    oof_predictions.columns
):
    raise KeyError(
        "OOF prediction file is missing required columns."
    )

if not grid["grid_id"].is_unique:
    raise ValueError(
        "Duplicate grid_id in the 500 m grid."
    )

if not feature_matrix["grid_id"].is_unique:
    raise ValueError(
        "Duplicate grid_id in the feature matrix."
    )

final_predictions = (
    oof_predictions[
        oof_predictions[
            "feature_set"
        ]
        == FINAL_FEATURE_SET
    ]
    .copy()
)

expected_validations = {
    "random_5fold",
    "spatial_kmeans_leave_one_out",
}

if set(
    final_predictions[
        "validation"
    ].unique()
) != expected_validations:
    print(
        "Warning: validation names found:",
        sorted(
            final_predictions[
                "validation"
            ].unique()
        ),
    )

for validation_name in (
    final_predictions[
        "validation"
    ].unique()
):
    subset = (
        final_predictions[
            final_predictions[
                "validation"
            ]
            == validation_name
        ]
    )

    if not subset[
        "grid_id"
    ].is_unique:
        raise ValueError(
            f"Duplicate OOF grid predictions for "
            f"{validation_name}."
        )

print("Grid cells:", len(grid))
print("Final OOF prediction rows:", len(final_predictions))
print("Available model summary:")
display(model_summary.round(4))

## 1. Final model-performance summary

The primary model is the tabular XGBoost model. The CNN configuration remains in the comparison table so the dissertation can report whether visual embeddings added incremental predictive value.

In [ ]:
# ============================================================
# Final model-performance table
# ============================================================

performance_columns = [
    "validation",
    "feature_set",
    "R2_mean",
    "R2_std",
    "RMSE_mean",
    "MAE_mean",
    "train_R2_mean",
    "OOF_R2",
    "OOF_RMSE",
    "OOF_MAE",
]

available_columns = [
    column
    for column in performance_columns
    if column in model_summary.columns
]

final_performance_table = (
    model_summary[
        available_columns
    ]
    .copy()
)

if {
    "train_R2_mean",
    "R2_mean",
}.issubset(
    final_performance_table.columns
):
    final_performance_table[
        "R2_generalisation_gap"
    ] = (
        final_performance_table[
            "train_R2_mean"
        ]
        - final_performance_table[
            "R2_mean"
        ]
    )

display(
    final_performance_table
    .round(4)
)

final_performance_table.to_csv(
    TABLE_DIR
    / "final_model_performance.csv",
    index=False,
)

if not cnn_gain.empty:
    print("\nIncremental effect of CNN features:")
    display(
        cnn_gain.round(4)
    )

## 2. Observed versus validated OOF prediction

The Random 5-fold OOF prediction is used for the main citywide predictive-performance scatter plot because each grid is predicted by a model that did not train on that grid.

In [ ]:
# ============================================================
# Observed versus predicted scatter plot
# ============================================================

random_oof = (
    final_predictions[
        final_predictions[
            "validation"
        ]
        == "random_5fold"
    ]
    .sort_values("grid_id")
    .reset_index(drop=True)
)

random_r2 = r2_score(
    random_oof["observed"],
    random_oof["predicted"],
)

random_rmse = np.sqrt(
    mean_squared_error(
        random_oof["observed"],
        random_oof["predicted"],
    )
)

random_mae = mean_absolute_error(
    random_oof["observed"],
    random_oof["predicted"],
)

axis_min = min(
    random_oof["observed"].min(),
    random_oof["predicted"].min(),
)

axis_max = max(
    random_oof["observed"].max(),
    random_oof["predicted"].max(),
)

fig, ax = plt.subplots(
    figsize=(6.5, 6.0)
)

ax.scatter(
    random_oof["observed"],
    random_oof["predicted"],
    alpha=0.45,
    s=22,
)

ax.plot(
    [axis_min, axis_max],
    [axis_min, axis_max],
    linestyle="--",
    linewidth=1.2,
)

ax.set_xlim(
    axis_min,
    axis_max,
)

ax.set_ylim(
    axis_min,
    axis_max,
)

ax.set_xlabel(
    "Observed electricity consumption"
)

ax.set_ylabel(
    "OOF-predicted electricity consumption"
)

ax.set_title(
    "Random 5-Fold OOF: Observed vs Predicted"
)

metric_text = (
    f"OOF R² = {random_r2:.3f}\n"
    f"RMSE = {random_rmse:.2f}\n"
    f"MAE = {random_mae:.2f}"
)

ax.text(
    0.04,
    0.96,
    metric_text,
    transform=ax.transAxes,
    va="top",
)

plt.tight_layout()

scatter_path = (
    FIGURE_DIR
    / "random_oof_observed_vs_predicted.png"
)

plt.savefig(
    scatter_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print("Saved:", scatter_path)

## 3. Build validated spatial prediction layers

Two independent OOF layers are created:

- **Random 5-fold OOF**: primary citywide validated prediction map.
- **Spatial K=4 OOF**: spatial-transferability map showing performance when complete spatial blocks are held out.

Observed and predicted maps use the same continuous scale within each validation scheme. Residual is defined as `observed - predicted`, so positive residuals indicate underprediction and negative residuals indicate overprediction.

In [ ]:
# ============================================================
# Merge Random and Spatial OOF predictions with the 500 m grid
# ============================================================

def build_prediction_gdf(
    validation_name,
):

    prediction_subset = (
        final_predictions[
            final_predictions[
                "validation"
            ]
            == validation_name
        ][
            [
                "grid_id",
                "fold",
                "observed",
                "predicted",
                "residual",
            ]
        ]
        .copy()
    )

    prediction_subset[
        "absolute_error"
    ] = prediction_subset[
        "residual"
    ].abs()

    prediction_gdf = (
        grid[
            [
                "grid_id",
                "geometry",
            ]
        ]
        .merge(
            prediction_subset,
            on="grid_id",
            how="inner",
            validate="one_to_one",
        )
    )

    return gpd.GeoDataFrame(
        prediction_gdf,
        geometry="geometry",
        crs=grid.crs,
    )


random_map_gdf = (
    build_prediction_gdf(
        "random_5fold"
    )
)

spatial_map_gdf = (
    build_prediction_gdf(
        "spatial_kmeans_leave_one_out"
    )
)

print(
    "Random OOF map cells:",
    len(random_map_gdf),
)

print(
    "Spatial OOF map cells:",
    len(spatial_map_gdf),
)

if len(random_map_gdf) != len(grid):
    print(
        "Warning: Random OOF map does not "
        "contain every grid."
    )

if len(spatial_map_gdf) != len(grid):
    print(
        "Warning: Spatial OOF map does not "
        "contain every grid."
    )

In [ ]:
# ============================================================
# Cell 1: Distribution comparison of observed and OOF predictions
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Use one observed series only
observed_series = (
    random_map_gdf[["grid_id", "observed"]]
    .drop_duplicates(subset="grid_id")
    .sort_values("grid_id")
    .reset_index(drop=True)
)

random_pred_series = (
    random_map_gdf[["grid_id", "predicted"]]
    .drop_duplicates(subset="grid_id")
    .sort_values("grid_id")
    .reset_index(drop=True)
    .rename(columns={"predicted": "value"})
)
random_pred_series["series"] = "Random OOF predicted"

spatial_pred_series = (
    spatial_map_gdf[["grid_id", "predicted"]]
    .drop_duplicates(subset="grid_id")
    .sort_values("grid_id")
    .reset_index(drop=True)
    .rename(columns={"predicted": "value"})
)
spatial_pred_series["series"] = "Spatial OOF predicted"

observed_plot_series = (
    observed_series
    .rename(columns={"observed": "value"})
    .copy()
)
observed_plot_series["series"] = "Observed"

distribution_df = pd.concat(
    [
        observed_plot_series[["grid_id", "value", "series"]],
        random_pred_series[["grid_id", "value", "series"]],
        spatial_pred_series[["grid_id", "value", "series"]],
    ],
    ignore_index=True,
)

print(
    distribution_df.groupby("series")["value"]
    .describe()[["count", "mean", "std", "min", "25%", "50%", "75%", "max"]]
    .round(2)
)

fig, axes = plt.subplots(
    1, 2, figsize=(16, 6)
)

# ------------------------------------------------------------
# Left: KDE only
# ------------------------------------------------------------

for series_name, subset in distribution_df.groupby("series"):

    sns.kdeplot(
        data=subset,
        x="value",
        ax=axes[0],
        linewidth=2,
        label=series_name,
    )

axes[0].set_title(
    "Distribution of observed and OOF-predicted electricity consumption"
)

axes[0].set_xlabel(
    "Electricity consumption"
)

axes[0].set_ylabel(
    "Density"
)

axes[0].legend(
    title="Series"
)

# ------------------------------------------------------------
# Right: Violin + Box
# ------------------------------------------------------------
sns.violinplot(
    data=distribution_df,
    x="series",
    y="value",
    inner=None,
    cut=0,
    ax=axes[1],
)

sns.boxplot(
    data=distribution_df,
    x="series",
    y="value",
    width=0.22,
    showcaps=True,
    boxprops={"facecolor": "white", "zorder": 3},
    whiskerprops={"zorder": 3},
    medianprops={"color": "black", "zorder": 4},
    flierprops={
        "marker": "o",
        "markersize": 2,
        "alpha": 0.35,
    },
    ax=axes[1],
)

axes[1].set_title(
    "Violin and box plot of observed and OOF-predicted values"
)
axes[1].set_xlabel("")
axes[1].set_ylabel("Electricity consumption")
axes[1].tick_params(axis="x", rotation=12)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 2: Log-scale distribution and upper-tail comparison
# ============================================================

# Use only positive values for the logarithmic x-axis
positive_distribution_df = (
    distribution_df[
        distribution_df["value"] > 0
    ]
    .copy()
)

# Define the upper tail using the observed 90th percentile
observed_values = (
    observed_plot_series["value"]
    .to_numpy()
)

tail_threshold = np.percentile(
    observed_values,
    90,
)

overall_max = (
    distribution_df["value"].max()
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(15, 5.5),
)

# ------------------------------------------------------------
# Left: Full distribution with logarithmic x-axis
# ------------------------------------------------------------

for series_name, subset in (
    positive_distribution_df.groupby("series")
):

    sns.kdeplot(
        data=subset,
        x="value",
        ax=axes[0],
        linewidth=2,
        label=series_name,
        common_norm=False,
    )

axes[0].set_xscale("log")

axes[0].set_title(
    "Distribution on Logarithmic Scale"
)

axes[0].set_xlabel(
    "Electricity consumption (log scale)"
)

axes[0].set_ylabel(
    "Density"
)

axes[0].legend(
    title="Series"
)

# ------------------------------------------------------------
# Right: Upper-tail detail
# ------------------------------------------------------------

for series_name, subset in (
    distribution_df.groupby("series")
):

    sns.kdeplot(
        data=subset,
        x="value",
        ax=axes[1],
        linewidth=2,
        label=series_name,
        common_norm=False,
    )

axes[1].axvline(
    tail_threshold,
    linestyle="--",
    linewidth=1,
)

axes[1].set_xlim(
    tail_threshold,
    overall_max,
)

axes[1].set_title(
    "Upper-Tail Distribution"
)

axes[1].set_xlabel(
    "Electricity consumption"
)

axes[1].set_ylabel(
    "Density"
)

axes[1].legend(
    title="Series"
)

axes[1].text(
    tail_threshold,
    axes[1].get_ylim()[1] * 0.95,
    f"Observed 90th percentile = {tail_threshold:.1f}",
    ha="left",
    va="top",
)

plt.tight_layout()

output_path = (
    FIGURE_DIR
    / "electricity_distribution_log_and_upper_tail.png"
)

plt.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print(
    "Observed 90th percentile:",
    round(tail_threshold, 2),
)

print(
    "Observed maximum:",
    round(
        observed_plot_series["value"].max(),
        2,
    ),
)

print(
    "Random OOF maximum:",
    round(
        random_pred_series["value"].max(),
        2,
    ),
)

print(
    "Spatial OOF maximum:",
    round(
        spatial_pred_series["value"].max(),
        2,
    ),
)

In [ ]:
# ============================================================
# Cell 2: Observed vs predicted scatter comparison
# ============================================================

observed_threshold = np.percentile(
    random_oof["observed"],
    98,
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 5.5),
    sharex=True,
    sharey=True,
)

plot_sets = [
    (
        random_oof,
        "Random 5-Fold OOF",
    ),
    (
        final_predictions[
            final_predictions["validation"]
            == "spatial_kmeans_leave_one_out"
        ],
        "Spatial K=4 OOF",
    ),
]

global_max = max(
    dataset[0][["observed", "predicted"]]
    .to_numpy()
    .max()
    for dataset in plot_sets
)

for ax, (
    data,
    title,
) in zip(
    axes,
    plot_sets,
):

    normal = (
        data["observed"]
        <= observed_threshold
    )

    high = ~normal

    ax.scatter(
        data.loc[
            normal,
            "observed",
        ],
        data.loc[
            normal,
            "predicted",
        ],
        s=18,
        alpha=0.35,
        label="Observed ≤ 98th percentile",
    )

    ax.scatter(
        data.loc[
            high,
            "observed",
        ],
        data.loc[
            high,
            "predicted",
        ],
        s=28,
        alpha=0.75,
        label="Observed > 98th percentile",
    )

    ax.plot(
        [0, global_max],
        [0, global_max],
        linestyle="--",
        linewidth=1.2,
        label="1:1 line",
    )

    ax.set_title(title)
    ax.set_xlabel(
        "Observed electricity consumption"
    )

    ax.set_xlim(
        0,
        global_max,
    )

    ax.set_ylim(
        0,
        global_max,
    )

axes[0].set_ylabel(
    "OOF-predicted electricity consumption"
)

axes[1].legend(
    loc="upper left"
)

plt.tight_layout()
plt.show()

print(
    "Observed 98th percentile:",
    round(
        observed_threshold,
        2,
    ),
)

In [ ]:
# ============================================================
# Check observed distribution tails
# ============================================================

percentiles = [
    0, 1, 2, 5,
    50,
    90, 95, 98, 99, 100,
]

values = random_oof["observed"]

print(
    values.quantile(
        np.array(percentiles) / 100
    )
)

p02 = values.quantile(0.02)
p98 = values.quantile(0.98)

print(
    "\nObserved < 2nd percentile:",
    (values < p02).sum(),
)

print(
    "Observed > 98th percentile:",
    (values > p98).sum(),
)

print(
    "2nd percentile:",
    round(p02, 2),
)

print(
    "98th percentile:",
    round(p98, 2),
)

In [ ]:
# ============================================================
# Shared percentile colour scales for Random and Spatial OOF maps
# ============================================================

from matplotlib.colors import Normalize, TwoSlopeNorm
from matplotlib.cm import ScalarMappable


# ------------------------------------------------------------
# Define one shared electricity scale across both validations
# ------------------------------------------------------------

electricity_values = np.concatenate([
    random_map_gdf["observed"].to_numpy(),
    random_map_gdf["predicted"].to_numpy(),
    spatial_map_gdf["predicted"].to_numpy(),
])

electricity_min = np.percentile(
    electricity_values,
    2,
)

electricity_max = np.percentile(
    electricity_values,
    98,
)

electricity_norm = Normalize(
    vmin=electricity_min,
    vmax=electricity_max,
    clip=True,
)


# ------------------------------------------------------------
# Define one shared residual scale across both validations
# ------------------------------------------------------------

all_abs_residuals = np.concatenate([
    random_map_gdf["residual"].abs().to_numpy(),
    spatial_map_gdf["residual"].abs().to_numpy(),
])

residual_limit = np.percentile(
    all_abs_residuals,
    95,
)

residual_norm = TwoSlopeNorm(
    vmin=-residual_limit,
    vcenter=0,
    vmax=residual_limit,
)


# ------------------------------------------------------------
# Mapping function
# ------------------------------------------------------------

def plot_oof_maps_shared_scale(
    prediction_gdf,
    title_prefix,
    output_filename,
):

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(18, 7),
    )

    prediction_gdf.plot(
        column="observed",
        ax=axes[0],
        cmap="viridis",
        norm=electricity_norm,
        edgecolor="none",
    )

    prediction_gdf.plot(
        column="predicted",
        ax=axes[1],
        cmap="viridis",
        norm=electricity_norm,
        edgecolor="none",
    )

    prediction_gdf.plot(
        column="residual",
        ax=axes[2],
        cmap="RdBu_r",
        norm=residual_norm,
        edgecolor="none",
    )

    axes[0].set_title(
        "Observed"
    )

    axes[1].set_title(
        "OOF Predicted"
    )

    axes[2].set_title(
        "Residual: Observed − Predicted"
    )

    for ax in axes:
        ax.set_axis_off()

    # Shared electricity colour bars
    electricity_sm = ScalarMappable(
        norm=electricity_norm,
        cmap="viridis",
    )

    fig.colorbar(
        electricity_sm,
        ax=axes[0],
        shrink=0.75,
        extend="both",
        label="Electricity consumption",
    )

    fig.colorbar(
        electricity_sm,
        ax=axes[1],
        shrink=0.75,
        extend="both",
        label="Electricity consumption",
    )

    # Shared residual colour bar
    residual_sm = ScalarMappable(
        norm=residual_norm,
        cmap="RdBu_r",
    )

    fig.colorbar(
        residual_sm,
        ax=axes[2],
        shrink=0.75,
        extend="both",
        label="Residual",
    )

    fig.suptitle(
        title_prefix,
        fontsize=14,
    )

    plt.tight_layout()

    output_path = (
        FIGURE_DIR
        / output_filename
    )

    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

    print(
        "Shared electricity range:",
        round(electricity_min, 2),
        "to",
        round(electricity_max, 2),
    )

    print(
        "Shared residual range:",
        round(-residual_limit, 2),
        "to",
        round(residual_limit, 2),
    )


# ------------------------------------------------------------
# Plot both validation schemes with identical colour scales
# ------------------------------------------------------------

plot_oof_maps_shared_scale(
    random_map_gdf,
    "Random 5-Fold OOF Prediction",
    "random_oof_maps_shared_scale.png",
)

plot_oof_maps_shared_scale(
    spatial_map_gdf,
    "Spatial K=4 OOF Prediction",
    "spatial_oof_maps_shared_scale.png",
)

In [ ]:
# ============================================================
# Prepare combined mapping dataframe and tail thresholds
# ============================================================

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, TwoSlopeNorm
from matplotlib.cm import ScalarMappable

base_map_gdf = random_map_gdf[
    ["grid_id", "observed", "geometry"]
].copy()

base_map_gdf = base_map_gdf.merge(
    random_map_gdf[
        ["grid_id", "predicted"]
    ].rename(
        columns={
            "predicted": "random_predicted"
        }
    ),
    on="grid_id",
    how="left",
)

base_map_gdf = base_map_gdf.merge(
    spatial_map_gdf[
        ["grid_id", "predicted"]
    ].rename(
        columns={
            "predicted": "spatial_predicted"
        }
    ),
    on="grid_id",
    how="left",
)

base_map_gdf["random_residual"] = (
    base_map_gdf["observed"]
    - base_map_gdf["random_predicted"]
)

base_map_gdf["spatial_residual"] = (
    base_map_gdf["observed"]
    - base_map_gdf["spatial_predicted"]
)

q02 = base_map_gdf["observed"].quantile(0.02)
q98 = base_map_gdf["observed"].quantile(0.98)

main_vmin = q02
main_vmax = q98

low_tail_mask = (
    base_map_gdf["observed"] < q02
)

high_tail_mask = (
    base_map_gdf["observed"] > q98
)

print(f"2nd percentile  = {q02:.2f}")
print(f"98th percentile = {q98:.2f}")
print(f"Low-tail grids   = {low_tail_mask.sum()}")
print(f"High-tail grids  = {high_tail_mask.sum()}")

In [ ]:
# ============================================================
# Main maps on the 2%-98% range
# ============================================================

def plot_main_maps_2_98(
    gdf,
    vmin,
    vmax,
    output_filename,
):
    fig, axes = plt.subplots(
        1,
        3,
        figsize=(18, 7),
    )

    plot_specs = [
        ("observed", "Observed"),
        ("random_predicted", "Random OOF predicted"),
        ("spatial_predicted", "Spatial OOF predicted"),
    ]

    for ax, (col, title) in zip(
        axes,
        plot_specs,
    ):
        gdf.plot(
            column=col,
            ax=ax,
            cmap="viridis",
            vmin=vmin,
            vmax=vmax,
            edgecolor="none",
            legend=True,
            legend_kwds={
                "label": "Electricity consumption",
                "shrink": 0.85,
            },
        )
        ax.set_title(title)
        ax.set_axis_off()

    fig.suptitle(
        "Main maps on the 2%–98% observed range",
        fontsize=14,
    )

    plt.tight_layout()

    output_path = (
        FIGURE_DIR
        / output_filename
    )
    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()

    print("Saved:", output_path)


plot_main_maps_2_98(
    base_map_gdf,
    vmin=main_vmin,
    vmax=main_vmax,
    output_filename="main_maps_2_98_range.png",
)

In [ ]:
# ============================================================
# Tail maps with colorbar moved to the far right
# ============================================================

def plot_tail_maps(
    gdf,
    mask,
    value_columns,
    titles,
    suptitle,
    output_filename,
    cmap,
):
    masked_gdf = gdf.copy()

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(20, 7),
    )

    # 先算三个面板共用的色阶
    tail_values = pd.concat(
        [
            masked_gdf.loc[mask, col]
            for col in value_columns
        ],
        axis=0,
    ).dropna()

    vmin = tail_values.min()
    vmax = tail_values.max()

    for i, (ax, col, title) in enumerate(
        zip(axes, value_columns, titles)
    ):
        # 背景：所有非目标格网淡灰显示
        masked_gdf.plot(
            ax=ax,
            color="white",
            edgecolor="#e6e6e6",
            linewidth=0.4,
        )

        # 仅绘制目标 tail 格网
        masked_gdf.loc[mask].plot(
            column=col,
            ax=ax,
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
            edgecolor="none",
            legend=False,
        )

        ax.set_title(title, fontsize=13)
        ax.set_axis_off()

    fig.suptitle(
        suptitle,
        fontsize=16,
        y=0.98,
    )

    # 把 colorbar 单独放到最右侧
    sm = plt.cm.ScalarMappable(
        cmap=cmap,
        norm=plt.Normalize(
            vmin=vmin,
            vmax=vmax,
        ),
    )
    sm._A = []

    cax = fig.add_axes(
        [0.92, 0.18, 0.012, 0.62]
    )  # left, bottom, width, height
    cbar = fig.colorbar(
        sm,
        cax=cax,
    )
    cbar.set_label(
        "Electricity consumption",
        fontsize=12,
    )

    plt.subplots_adjust(
        left=0.03,
        right=0.90,
        top=0.90,
        bottom=0.06,
        wspace=0.12,
    )

    output_path = (
        FIGURE_DIR / output_filename
    )

    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()

    print("Saved:", output_path)

In [ ]:
plot_tail_maps(
    gdf=base_map_gdf,
    mask=low_tail_mask,
    value_columns=[
        "observed",
        "random_predicted",
        "spatial_predicted",
    ],
    titles=[
        "Observed < 2%",
        "Random OOF predicted",
        "Spatial OOF predicted",
    ],
    suptitle="Low-tail grids: Observed < 2nd percentile",
    output_filename="low_tail_maps_blue.png",
    cmap="Blues",
)

plot_tail_maps(
    gdf=base_map_gdf,
    mask=high_tail_mask,
    value_columns=[
        "observed",
        "random_predicted",
        "spatial_predicted",
    ],
    titles=[
        "Observed > 98%",
        "Random OOF predicted",
        "Spatial OOF predicted",
    ],
    suptitle="High-tail grids: Observed > 98th percentile",
    output_filename="high_tail_maps_red.png",
    cmap="Reds",
)

In [ ]:
# ============================================================
# Export low-tail and high-tail grid lists
# ============================================================

low_tail_table = base_map_gdf.loc[
    low_tail_mask,
    [
        "grid_id",
        "observed",
        "random_predicted",
        "spatial_predicted",
        "random_residual",
        "spatial_residual",
    ],
].sort_values(
    "observed",
    ascending=True,
)

high_tail_table = base_map_gdf.loc[
    high_tail_mask,
    [
        "grid_id",
        "observed",
        "random_predicted",
        "spatial_predicted",
        "random_residual",
        "spatial_residual",
    ],
].sort_values(
    "observed",
    ascending=False,
)

low_tail_path = (
    FIGURE_DIR
    / "low_tail_grid_list.csv"
)

high_tail_path = (
    FIGURE_DIR
    / "high_tail_grid_list.csv"
)

low_tail_table.to_csv(
    low_tail_path,
    index=False,
)

high_tail_table.to_csv(
    high_tail_path,
    index=False,
)

display(low_tail_table.head(10))
display(high_tail_table.head(10))

print("Saved:", low_tail_path)
print("Saved:", high_tail_path)

## 4. Spatial-block error analysis

This section explains spatial generalisation rather than only reporting one pooled metric. It is useful for identifying which held-out spatial region is difficult and whether the problem is associated with different target distributions.

In [ ]:
# ============================================================
# Spatial-fold and spatial-block error analysis
# ============================================================

spatial_analysis = (
    final_predictions[
        final_predictions[
            "validation"
        ]
        == "spatial_kmeans_leave_one_out"
    ]
    .copy()
)

spatial_analysis[
    "absolute_error"
] = (
    spatial_analysis[
        "residual"
    ].abs()
)

spatial_analysis = (
    spatial_analysis.merge(
        spatial_blocks[
            [
                "grid_id",
                "spatial_block",
            ]
        ],
        on="grid_id",
        how="left",
        validate="one_to_one",
    )
)

spatial_error_rows = []

for (
    fold,
    block_id,
), group in (
    spatial_analysis.groupby(
        [
            "fold",
            "spatial_block",
        ]
    )
):

    spatial_error_rows.append({
        "fold": fold,
        "spatial_block": block_id,
        "n_grids": len(group),
        "observed_mean": (
            group["observed"].mean()
        ),
        "observed_std": (
            group["observed"].std()
        ),
        "predicted_mean": (
            group["predicted"].mean()
        ),
        "R2": r2_score(
            group["observed"],
            group["predicted"],
        ),
        "RMSE": np.sqrt(
            mean_squared_error(
                group["observed"],
                group["predicted"],
            )
        ),
        "MAE": mean_absolute_error(
            group["observed"],
            group["predicted"],
        ),
        "mean_residual": (
            group["residual"].mean()
        ),
    })

spatial_error_table = pd.DataFrame(
    spatial_error_rows
).sort_values(
    "fold"
)

display(
    spatial_error_table.round(4)
)

spatial_error_table.to_csv(
    TABLE_DIR
    / "spatial_block_error_analysis.csv",
    index=False,
)

fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.bar(
    spatial_error_table[
        "fold"
    ].astype(str),
    spatial_error_table[
        "R2"
    ],
)

ax.axhline(
    0,
    linewidth=0.8,
)

ax.set_xlabel(
    "Spatial outer fold"
)

ax.set_ylabel(
    "R²"
)

ax.set_title(
    "Spatial OOF Performance by Held-Out Block"
)

plt.tight_layout()

block_plot_path = (
    FIGURE_DIR
    / "spatial_fold_r2.png"
)

plt.savefig(
    block_plot_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## 5. Prediction-error structure

This section tests whether errors increase systematically with electricity consumption and whether high-consumption grids tend to be underpredicted.

In [ ]:
# ============================================================
# Error analysis for the primary Random 5-fold OOF predictions
# ============================================================

error_df = random_oof.copy()

error_df[
    "absolute_error"
] = (
    error_df[
        "residual"
    ].abs()
)

error_df[
    "observed_decile"
] = pd.qcut(
    error_df["observed"],
    q=10,
    duplicates="drop",
)

error_by_decile = (
    error_df
    .groupby(
        "observed_decile",
        observed=True,
    )
    .agg(
        n_grids=(
            "grid_id",
            "count",
        ),
        observed_mean=(
            "observed",
            "mean",
        ),
        predicted_mean=(
            "predicted",
            "mean",
        ),
        mean_residual=(
            "residual",
            "mean",
        ),
        MAE=(
            "absolute_error",
            "mean",
        ),
    )
    .reset_index()
)

display(
    error_by_decile.round(4)
)

error_by_decile.to_csv(
    TABLE_DIR
    / "random_oof_error_by_observed_decile.csv",
    index=False,
)

fig, ax = plt.subplots(
    figsize=(7, 5)
)

ax.scatter(
    error_df[
        "observed"
    ],
    error_df[
        "absolute_error"
    ],
    alpha=0.40,
    s=20,
)

ax.set_xlabel(
    "Observed electricity consumption"
)

ax.set_ylabel(
    "Absolute OOF prediction error"
)

ax.set_title(
    "Prediction Error vs Observed Consumption"
)

plt.tight_layout()

error_plot_path = (
    FIGURE_DIR
    / "absolute_error_vs_observed.png"
)

plt.savefig(
    error_plot_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

residual_correlation = (
    error_df[
        [
            "observed",
            "absolute_error",
        ]
    ]
    .corr(
        method="spearman"
    )
    .loc[
        "observed",
        "absolute_error",
    ]
)

print(
    "Spearman correlation between observed "
    "consumption and absolute error:",
    round(
        residual_correlation,
        4,
    ),
)

In [ ]:
# ============================================================
# Observed vs predicted electricity consumption
# Random 5-fold OOF with 1:1 reference line
# ============================================================

fig, ax = plt.subplots(
    figsize=(6, 6)
)

ax.scatter(
    random_oof["observed"],
    random_oof["predicted"],
    alpha=0.40,
    s=20,
)

# ------------------------------------------------------------
# 1:1 reference line
# ------------------------------------------------------------

axis_min = min(
    random_oof["observed"].min(),
    random_oof["predicted"].min(),
)

axis_max = max(
    random_oof["observed"].max(),
    random_oof["predicted"].max(),
)

ax.plot(
    [axis_min, axis_max],
    [axis_min, axis_max],
    linestyle="--",
    linewidth=1.5,
    label="1:1 line",
)

# Same scale on both axes
ax.set_xlim(
    axis_min,
    axis_max,
)

ax.set_ylim(
    axis_min,
    axis_max,
)

ax.set_aspect(
    "equal",
    adjustable="box",
)

# ------------------------------------------------------------
# Labels
# ------------------------------------------------------------

ax.set_xlabel(
    "Observed electricity consumption (kWh/month)"
)

ax.set_ylabel(
    "Predicted electricity consumption (kWh/month)"
)

ax.set_title(
    "Observed vs Predicted Electricity Consumption"
)

ax.legend(
    frameon=False
)

plt.tight_layout()

prediction_plot_path = (
    FIGURE_DIR
    / "observed_vs_predicted_random_oof.png"
)

plt.savefig(
    prediction_plot_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## 6. Final tabular-model interpretation

For interpretation only, a single XGBoost model is refitted on all labelled grids. To avoid selecting a configuration based on outer-test performance, the hyperparameter set is taken from the **Random 5-fold Tabular outer fold with the highest inner-CV R²**.

This full-data model is **not** used to replace the OOF performance metrics or OOF prediction maps.

In [ ]:
# ============================================================
# Prepare the final explanatory XGBoost model
# ============================================================

TARGET_COLUMN = "elec_consumption"
INNER_SCORE_COLUMN = "inner_best_R2_fit_scale"

# Features excluded from the final 36-feature baseline
FEATURES_TO_DROP = [
    "brightness",
    "poi_total_count",
    "pop_total",
    "road_length_total",
    "building_area_total",
    "NTL_std",
]

# Final 36 tabular predictors
BASELINE_FEATURES = [
    "area_m2",

    "B02_mean", "B02_std", "B02_max",
    "B03_mean", "B03_std", "B03_max",
    "B04_mean", "B04_std", "B04_max",
    "B08_mean", "B08_std", "B08_max",
    "B11_mean", "B11_std", "B11_max",

    "NDVI", "NDBI",

    "NTL_mean", "NTL_max",

    "dist_major_road",
    "road_count",
    "road_length_major",
    "road_density",
    "major_road_ratio",

    "poi_economic_count",
    "poi_social_count",
    "poi_other_count",
    "poi_shannon",
    "dist_economic_poi",
    "dist_social_poi",

    "pop_density_km2",

    "building_count",
    "building_area_mean",
    "building_area_std",
    "building_coverage",
]

assert len(BASELINE_FEATURES) == 36


# ============================================================
# Prepare final tabular feature matrix
# ============================================================

working_features = (
    feature_matrix
    .drop(
        columns=[
            column
            for column in FEATURES_TO_DROP
            if column in feature_matrix.columns
        ]
    )
    .copy()
)

missing_features = [
    column
    for column in BASELINE_FEATURES
    if column not in working_features.columns
]

if missing_features:
    raise KeyError(
        f"Missing baseline features: {missing_features}"
    )

if TARGET_COLUMN not in working_features.columns:
    raise KeyError(
        f"Target not found: {TARGET_COLUMN}"
    )


# ============================================================
# Retrieve the final Random-CV tabular results from notebook 003
# ============================================================

tabular_random_folds = (
    fold_results[
        (fold_results["validation"] == "random_5fold")
        & (fold_results["feature_set"] == "Tabular")
        & (fold_results["target_mode"] == "raw")
    ]
    .copy()
)

if tabular_random_folds.empty:
    raise ValueError(
        "No raw-target Random 5-fold Tabular results "
        "were found in the 003 fold table."
    )

if INNER_SCORE_COLUMN not in tabular_random_folds.columns:
    raise KeyError(
        f"Required tuning-score column not found: "
        f"{INNER_SCORE_COLUMN}"
    )

print(
    "Available explanatory-model folds:",
    len(tabular_random_folds),
)

display(
    tabular_random_folds[
        [
            "validation",
            "feature_set",
            "target_mode",
            "fold",
            "R2",
            "RMSE",
            "MAE",
            "train_R2",
            INNER_SCORE_COLUMN,
            "best_params",
        ]
    ]
)


# ============================================================
# Select the strongest inner-CV parameter configuration
# ============================================================

selected_parameter_row = (
    tabular_random_folds
    .sort_values(
        INNER_SCORE_COLUMN,
        ascending=False,
    )
    .iloc[0]
)

raw_params = selected_parameter_row["best_params"]

if isinstance(raw_params, str):
    selected_params = json.loads(raw_params)
else:
    selected_params = dict(raw_params)

xgb_params = {
    key.replace("model__", ""): value
    for key, value in selected_params.items()
    if key.startswith("model__")
}

xgb_params.update({
    "objective": "reg:squarederror",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbosity": 0,
})


# ============================================================
# Fit the full-data explanatory model
# ============================================================

X_full = (
    working_features[
        BASELINE_FEATURES
    ]
    .copy()
)

y_full = (
    working_features[
        TARGET_COLUMN
    ]
    .to_numpy(dtype=float)
)

final_imputer = SimpleImputer(
    strategy="median"
)

X_full_imputed = (
    final_imputer
    .fit_transform(
        X_full
    )
)

final_explanatory_model = XGBRegressor(
    **xgb_params
)

final_explanatory_model.fit(
    X_full_imputed,
    y_full,
)


# ============================================================
# Summary
# ============================================================

print(
    "Final explanatory predictors:",
    len(BASELINE_FEATURES),
)

print(
    "Selected source fold:",
    int(selected_parameter_row["fold"]),
)

print(
    "Selected target mode:",
    selected_parameter_row["target_mode"],
)

print(
    "Selected inner-CV R²:",
    round(
        selected_parameter_row[
            INNER_SCORE_COLUMN
        ],
        4,
    ),
)

print(
    "Explanatory model parameters:"
)

print(
    json.dumps(
        xgb_params,
        indent=2,
        default=str,
    )
)

In [ ]:
# ============================================================
# XGBoost gain-based feature importance
# ============================================================

booster = (
    final_explanatory_model
    .get_booster()
)

raw_gain = booster.get_score(
    importance_type="gain"
)

feature_gain_rows = []

for index, feature_name in enumerate(
    BASELINE_FEATURES
):

    model_feature_name = (
        f"f{index}"
    )

    feature_gain_rows.append({
        "feature": feature_name,
        "gain": raw_gain.get(
            model_feature_name,
            0.0,
        ),
    })

feature_importance = (
    pd.DataFrame(
        feature_gain_rows
    )
    .sort_values(
        "gain",
        ascending=False,
    )
    .reset_index(drop=True)
)

gain_total = (
    feature_importance[
        "gain"
    ].sum()
)

if gain_total > 0:
    feature_importance[
        "gain_share"
    ] = (
        feature_importance[
            "gain"
        ]
        / gain_total
    )
else:
    feature_importance[
        "gain_share"
    ] = 0.0

display(
    feature_importance
    .head(15)
    .round(4)
)

feature_importance.to_csv(
    TABLE_DIR
    / "xgboost_gain_feature_importance.csv",
    index=False,
)

top_importance = (
    feature_importance
    .head(15)
    .sort_values(
        "gain",
        ascending=True,
    )
)

fig, ax = plt.subplots(
    figsize=(8, 6)
)

ax.barh(
    top_importance[
        "feature"
    ],
    top_importance[
        "gain"
    ],
)

ax.set_xlabel(
    "XGBoost gain"
)

ax.set_ylabel(
    "Feature"
)

ax.set_title(
    "Top 15 Features in the Final Tabular XGBoost Model"
)

plt.tight_layout()

importance_path = (
    FIGURE_DIR
    / "xgboost_gain_feature_importance.png"
)

plt.savefig(
    importance_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print("Saved:", importance_path)

In [ ]:
# ============================================================
# SHAP analysis using XGBoost native TreeSHAP
# ============================================================

import shap
import xgboost as xgb

# Build an XGBoost DMatrix from the final tabular data
X_shap = xgb.DMatrix(
    X_full_imputed,
    feature_names=BASELINE_FEATURES,
)

# Calculate exact TreeSHAP contributions
shap_contributions = (
    final_explanatory_model
    .get_booster()
    .predict(
        X_shap,
        pred_contribs=True,
    )
)

# The final column is the expected value / bias term
shap_values = (
    shap_contributions[:, :-1]
)

base_values = (
    shap_contributions[:, -1]
)

print(
    "SHAP values shape:",
    shap_values.shape,
)

print(
    "Mean base value:",
    base_values.mean(),
)

# SHAP summary plot
shap.summary_plot(
    shap_values,
    X_full_imputed,
    feature_names=BASELINE_FEATURES,
    max_display=15,
    show=False,
)

plt.tight_layout()

shap_path = (
    FIGURE_DIR
    / "shap_summary_top15.png"
)

plt.savefig(
    shap_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

# Calculate mean absolute SHAP importance
mean_abs_shap = (
    np.abs(
        shap_values
    )
    .mean(axis=0)
)

shap_importance = (
    pd.DataFrame({
        "feature": BASELINE_FEATURES,
        "mean_abs_shap": mean_abs_shap,
    })
    .sort_values(
        "mean_abs_shap",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    shap_importance
    .head(15)
    .round(4)
)

shap_importance.to_csv(
    TABLE_DIR
    / "shap_feature_importance.csv",
    index=False,
)

print(
    "Saved:",
    shap_path,
)

In [ ]:
# ============================================================
# Prepare spatial analysis dataframe
# ============================================================

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path

grid_features_gdf = gpd.read_file(
    PROJECT_DATA_DIR
    / "grid_size_selection/grids/grid_features_500m.gpkg"
)

analysis_gdf = grid_features_gdf[
    [
        "grid_id",
        "pop_density_km2",
        "building_coverage",
        "road_density",
        "area_m2",
        "geometry",
    ]
].copy()

# Add population total if available
if "pop_total" in grid_features_gdf.columns:
    analysis_gdf["pop_total"] = grid_features_gdf["pop_total"]
else:
    analysis_gdf["pop_total"] = (
        analysis_gdf["pop_density_km2"]
        * analysis_gdf["area_m2"]
        / 1_000_000
    )

# Add Random OOF predictions
analysis_gdf = analysis_gdf.merge(
    random_map_gdf[
        [
            "grid_id",
            "observed",
            "predicted",
        ]
    ].rename(
        columns={
            "predicted": "random_predicted"
        }
    ),
    on="grid_id",
    how="left",
)

analysis_gdf["residual"] = (
    analysis_gdf["observed"]
    - analysis_gdf["random_predicted"]
)

print("Rows:", len(analysis_gdf))
analysis_gdf.head()

In [ ]:
# ============================================================
# Grid choropleth: observed vs Random OOF predicted electricity
# ============================================================

from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
import matplotlib.pyplot as plt
import numpy as np

map_gdf = analysis_gdf.copy()

# Use observed 2nd-98th percentiles as one shared colour scale
vmin = map_gdf["observed"].quantile(0.02)
vmax = map_gdf["observed"].quantile(0.98)

norm = Normalize(
    vmin=vmin,
    vmax=vmax,
    clip=True,
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 7),
)

plot_specs = [
    ("observed", "Observed electricity consumption"),
    ("random_predicted", "Random OOF predicted electricity consumption"),
]

for ax, (column, title) in zip(
    axes,
    plot_specs,
):
    map_gdf.plot(
        column=column,
        ax=ax,
        cmap="Reds",
        norm=norm,
        edgecolor="none",
    )

    ax.set_title(title)
    ax.set_axis_off()

# Shared colourbar
sm = ScalarMappable(
    norm=norm,
    cmap="Reds",
)

cax = fig.add_axes(
    [0.92, 0.18, 0.015, 0.64]
)

cbar = fig.colorbar(
    sm,
    cax=cax,
    extend="both",
)

cbar.set_label(
    "Electricity consumption"
)

plt.subplots_adjust(
    right=0.90,
    wspace=0.05,
)

output_path = (
    FIGURE_DIR
    / "observed_vs_random_oof_choropleth.png"
)

plt.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
# ============================================================
# Recreate Random and Spatial OOF dataframes
# without rerunning the whole notebook
# ============================================================

# If final_predictions is still in memory, use it directly
if "final_predictions" in globals():

    random_oof = (
        final_predictions[
            final_predictions["validation"]
            == "random_5fold"
        ]
        .sort_values("grid_id")
        .reset_index(drop=True)
        .copy()
    )

    spatial_oof = (
        final_predictions[
            final_predictions["validation"]
            == "spatial_kmeans_leave_one_out"
        ]
        .sort_values("grid_id")
        .reset_index(drop=True)
        .copy()
    )

else:
    # Fallback: reload only the latest 003 OOF prediction CSV
    oof_predictions = pd.read_csv(
        OOF_PREDICTION_PATH
    )

    final_predictions = (
        oof_predictions[
            oof_predictions["feature_set"] == "Tabular"
        ]
        .copy()
    )

    # Keep raw target if this column exists
    if "target_mode" in final_predictions.columns:
        final_predictions = (
            final_predictions[
                final_predictions["target_mode"] == "raw"
            ]
            .copy()
        )

    random_oof = (
        final_predictions[
            final_predictions["validation"]
            == "random_5fold"
        ]
        .sort_values("grid_id")
        .reset_index(drop=True)
        .copy()
    )

    spatial_oof = (
        final_predictions[
            final_predictions["validation"]
            == "spatial_kmeans_leave_one_out"
        ]
        .sort_values("grid_id")
        .reset_index(drop=True)
        .copy()
    )


# ------------------------------------------------------------
# Quick checks
# ------------------------------------------------------------

print("Random OOF rows:", len(random_oof))
print("Spatial OOF rows:", len(spatial_oof))

print(
    "Random unique grids:",
    random_oof["grid_id"].nunique()
)

print(
    "Spatial unique grids:",
    spatial_oof["grid_id"].nunique()
)

In [ ]:
# ============================================================
# Grid choropleth:
# Observed vs Random OOF vs Spatial OOF electricity
# ============================================================

from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
import matplotlib.pyplot as plt
import numpy as np


# ------------------------------------------------------------
# Prepare mapping dataframe
# ------------------------------------------------------------

map_gdf = analysis_gdf.copy()

# Add Spatial OOF predictions if not already present
if "spatial_predicted" not in map_gdf.columns:

    spatial_map = (
        spatial_oof[
            ["grid_id", "predicted"]
        ]
        .rename(
            columns={
                "predicted": "spatial_predicted"
            }
        )
        .copy()
    )

    map_gdf = map_gdf.merge(
        spatial_map,
        on="grid_id",
        how="left",
    )


# ------------------------------------------------------------
# Use one shared colour scale
# based on observed 2nd–98th percentiles
# ------------------------------------------------------------

vmin = map_gdf[
    "observed"
].quantile(0.02)

vmax = map_gdf[
    "observed"
].quantile(0.98)

norm = Normalize(
    vmin=vmin,
    vmax=vmax,
    clip=True,
)


# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 7),
)

plot_specs = [
    (
        "observed",
        "Observed electricity consumption",
    ),
    (
        "random_predicted",
        "Random OOF prediction",
    ),
    (
        "spatial_predicted",
        "Spatial OOF prediction",
    ),
]

for ax, (column, title) in zip(
    axes,
    plot_specs,
):

    map_gdf.plot(
        column=column,
        ax=ax,
        cmap="Reds",
        norm=norm,
        edgecolor="none",
    )

    ax.set_title(
        title,
        fontsize=12,
    )

    ax.set_axis_off()


# ------------------------------------------------------------
# Shared colourbar
# ------------------------------------------------------------

sm = ScalarMappable(
    norm=norm,
    cmap="Reds",
)

sm.set_array([])

cax = fig.add_axes(
    [0.92, 0.18, 0.015, 0.64]
)

cbar = fig.colorbar(
    sm,
    cax=cax,
    extend="both",
)

cbar.set_label(
    "Electricity consumption (kWh/month)"
)


# ------------------------------------------------------------
# Layout
# ------------------------------------------------------------

plt.subplots_adjust(
    right=0.90,
    wspace=0.03,
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

output_path = (
    FIGURE_DIR
    / "observed_random_spatial_oof_choropleth.png"
)

plt.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print(
    f"Saved: {output_path}"
)

In [ ]:
# ============================================================
# Observed vs predicted Getis-Ord Gi* analysis
# ============================================================

from libpysal.weights import Queen
from esda.getisord import G_Local
from statsmodels.stats.multitest import multipletests


# ============================================================
# Prepare identical grids for observed and predicted Gi*
# ============================================================

gi_gdf = (
    analysis_gdf[
        [
            "grid_id",
            "observed",
            "random_predicted",
            "geometry",
        ]
    ]
    .dropna()
    .set_index("grid_id")
    .copy()
)

# Build Queen-contiguity weights
w_initial = Queen.from_dataframe(
    gi_gdf,
    use_index=True,
)

# Exclude islands so both surfaces use exactly the same grids
islands = list(
    w_initial.islands
)

if len(islands) > 0:
    print(
        "Isolated grids excluded from Gi*:",
        len(islands),
    )

gi_valid = (
    gi_gdf
    .drop(
        index=islands
    )
    .copy()
)

# Rebuild weights after removing islands
w = Queen.from_dataframe(
    gi_valid,
    use_index=True,
)

w.transform = "R"


# ============================================================
# Function for Gi* classification
# ============================================================

def calculate_gistar(
    values,
    weights,
    seed=42,
):

    gi_result = G_Local(
        values,
        weights,
        transform="R",
        permutations=999,
        star=True,
        seed=seed,
    )

    z_scores = gi_result.Zs
    p_raw = gi_result.p_sim

    # FDR correction for multiple local tests
    p_fdr = multipletests(
        p_raw,
        alpha=0.05,
        method="fdr_bh",
    )[1]

    classification = np.full(
        len(values),
        "Not significant",
        dtype=object,
    )

    classification[
        (z_scores > 0)
        & (p_fdr < 0.05)
    ] = "Hot spot"

    classification[
        (z_scores < 0)
        & (p_fdr < 0.05)
    ] = "Cold spot"

    return (
        z_scores,
        p_raw,
        p_fdr,
        classification,
    )


# ============================================================
# Observed Gi*
# ============================================================

(
    gi_valid["observed_GiZ"],
    gi_valid["observed_p_raw"],
    gi_valid["observed_p_fdr"],
    gi_valid["observed_gi_class"],
) = calculate_gistar(
    gi_valid["observed"].to_numpy(),
    w,
    seed=RANDOM_STATE,
)


# ============================================================
# Predicted Gi*
# ============================================================

(
    gi_valid["predicted_GiZ"],
    gi_valid["predicted_p_raw"],
    gi_valid["predicted_p_fdr"],
    gi_valid["predicted_gi_class"],
) = calculate_gistar(
    gi_valid["random_predicted"].to_numpy(),
    w,
    seed=RANDOM_STATE,
)


# ============================================================
# Hotspot and coldspot overlap metrics
# ============================================================

def calculate_overlap_metrics(
    observed_class,
    predicted_class,
    class_name,
):

    observed_mask = (
        observed_class == class_name
    )

    predicted_mask = (
        predicted_class == class_name
    )

    overlap_mask = (
        observed_mask
        & predicted_mask
    )

    union_mask = (
        observed_mask
        | predicted_mask
    )

    n_observed = int(
        observed_mask.sum()
    )

    n_predicted = int(
        predicted_mask.sum()
    )

    n_overlap = int(
        overlap_mask.sum()
    )

    n_union = int(
        union_mask.sum()
    )

    return {
        "Class": class_name,

        "Observed grids":
            n_observed,

        "Predicted grids":
            n_predicted,

        "Overlap grids":
            n_overlap,

        "Jaccard":
            (
                n_overlap / n_union
                if n_union > 0
                else np.nan
            ),

        "Observed recovery":
            (
                n_overlap / n_observed
                if n_observed > 0
                else np.nan
            ),

        "Predicted confirmation":
            (
                n_overlap / n_predicted
                if n_predicted > 0
                else np.nan
            ),
    }


gi_overlap_table = pd.DataFrame([
    calculate_overlap_metrics(
        gi_valid["observed_gi_class"],
        gi_valid["predicted_gi_class"],
        "Hot spot",
    ),

    calculate_overlap_metrics(
        gi_valid["observed_gi_class"],
        gi_valid["predicted_gi_class"],
        "Cold spot",
    ),
])

display(
    gi_overlap_table.round(4)
)

gi_overlap_table.to_csv(
    TABLE_DIR
    / "observed_predicted_gistar_overlap.csv",
    index=False,
)


print("\nObserved Gi* classes:")
print(
    gi_valid[
        "observed_gi_class"
    ].value_counts()
)

print("\nPredicted Gi* classes:")
print(
    gi_valid[
        "predicted_gi_class"
    ].value_counts()
)

In [ ]:
# ============================================================
# Visualise observed and predicted Gi* classes
# ============================================================

from matplotlib.patches import Patch

gi_plot_gdf = analysis_gdf.copy()

# Join Gi* results back to the full analysis grid
gi_plot_gdf = gi_plot_gdf.merge(
    gi_valid[
        [
            "observed_gi_class",
            "predicted_gi_class",
        ]
    ],
    left_on="grid_id",
    right_index=True,
    how="left",
)

gi_class_colors = {
    "Cold spot": "#2166ac",
    "Not significant": "#eeeeee",
    "Hot spot": "#b2182b",
}

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 7),
)

plot_specs = [
    (
        "observed_gi_class",
        "Observed electricity Gi*",
    ),
    (
        "predicted_gi_class",
        "Predicted electricity Gi*",
    ),
]

for ax, (column, title) in zip(
    axes,
    plot_specs,
):

    # Base layer
    gi_plot_gdf.plot(
        ax=ax,
        color="white",
        edgecolor="#d9d9d9",
        linewidth=0.2,
    )

    # Plot each Gi* class
    for category in [
        "Cold spot",
        "Not significant",
        "Hot spot",
    ]:

        subset = gi_plot_gdf[
            gi_plot_gdf[column] == category
        ]

        if not subset.empty:
            subset.plot(
                ax=ax,
                color=gi_class_colors[category],
                edgecolor="none",
            )

    ax.set_title(title)
    ax.set_axis_off()


# ============================================================
# Manual legend
# ============================================================

legend_handles = [
    Patch(
        facecolor=gi_class_colors["Hot spot"],
        edgecolor="none",
        label="Hot spot",
    ),
    Patch(
        facecolor=gi_class_colors["Cold spot"],
        edgecolor="none",
        label="Cold spot",
    ),
    Patch(
        facecolor=gi_class_colors["Not significant"],
        edgecolor="none",
        label="Not significant",
    ),
]

axes[1].legend(
    handles=legend_handles,
    title="Gi* class",
    loc="upper right",
    frameon=True,
)


# ============================================================
# Save figure
# ============================================================

plt.tight_layout()

output_path = (
    FIGURE_DIR
    / "observed_vs_predicted_gistar_maps.png"
)

plt.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print("Saved:", output_path)

In [ ]:
# ============================================================
# Observed vs predicted Gi* agreement map
# ============================================================

from matplotlib.patches import Patch

# ============================================================
# Build agreement classes
# ============================================================

gi_plot_gdf["gi_agreement"] = "Other / mismatch"

# Hot spot overlap
gi_plot_gdf.loc[
    (
        gi_plot_gdf["observed_gi_class"] == "Hot spot"
    )
    & (
        gi_plot_gdf["predicted_gi_class"] == "Hot spot"
    ),
    "gi_agreement",
] = "Hot spot overlap"

# Cold spot overlap
gi_plot_gdf.loc[
    (
        gi_plot_gdf["observed_gi_class"] == "Cold spot"
    )
    & (
        gi_plot_gdf["predicted_gi_class"] == "Cold spot"
    ),
    "gi_agreement",
] = "Cold spot overlap"

# Both not significant
gi_plot_gdf.loc[
    (
        gi_plot_gdf["observed_gi_class"] == "Not significant"
    )
    & (
        gi_plot_gdf["predicted_gi_class"] == "Not significant"
    ),
    "gi_agreement",
] = "Both not significant"


# ============================================================
# Colours
# ============================================================

agreement_colors = {
    "Hot spot overlap": "#b2182b",
    "Cold spot overlap": "#2166ac",
    "Both not significant": "#eeeeee",
    "Other / mismatch": "#f4a261",
}


# ============================================================
# Plot
# ============================================================

fig, ax = plt.subplots(
    figsize=(9, 8)
)

plot_order = [
    "Both not significant",
    "Other / mismatch",
    "Cold spot overlap",
    "Hot spot overlap",
]

for category in plot_order:

    subset = gi_plot_gdf[
        gi_plot_gdf["gi_agreement"] == category
    ]

    if not subset.empty:
        subset.plot(
            ax=ax,
            color=agreement_colors[category],
            edgecolor="none",
        )

ax.set_title(
    "Observed vs Predicted Gi* Agreement"
)

ax.set_axis_off()


# ============================================================
# Manual legend
# ============================================================

legend_handles = [
    Patch(
        facecolor=agreement_colors["Hot spot overlap"],
        edgecolor="none",
        label="Hot spot overlap",
    ),
    Patch(
        facecolor=agreement_colors["Cold spot overlap"],
        edgecolor="none",
        label="Cold spot overlap",
    ),
    Patch(
        facecolor=agreement_colors["Both not significant"],
        edgecolor="none",
        label="Both not significant",
    ),
    Patch(
        facecolor=agreement_colors["Other / mismatch"],
        edgecolor="none",
        label="Other / mismatch",
    ),
]

ax.legend(
    handles=legend_handles,
    title="Agreement class",
    loc="upper right",
    frameon=True,
)


# ============================================================
# Save
# ============================================================

plt.tight_layout()

output_path = (
    FIGURE_DIR
    / "gistar_agreement_map.png"
)

plt.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print("Saved:", output_path)

In [ ]:
# ============================================================
# Hot spot and cold spot overlap maps
# ============================================================

gi_plot_gdf["hotspot_overlap"] = (
    (gi_plot_gdf["observed_gi_class"] == "Hot spot")
    & (gi_plot_gdf["predicted_gi_class"] == "Hot spot")
)

gi_plot_gdf["coldspot_overlap"] = (
    (gi_plot_gdf["observed_gi_class"] == "Cold spot")
    & (gi_plot_gdf["predicted_gi_class"] == "Cold spot")
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 7),
)

overlap_specs = [
    ("hotspot_overlap", "Hot spot overlap"),
    ("coldspot_overlap", "Cold spot overlap"),
]

for ax, (column, title) in zip(axes, overlap_specs):

    gi_plot_gdf.plot(
        ax=ax,
        color="#f2f2f2",
        edgecolor="white",
        linewidth=0.1,
    )

    gi_plot_gdf.loc[
        gi_plot_gdf[column]
    ].plot(
        ax=ax,
        color="#e75480",
        edgecolor="none",
    )

    ax.set_title(title)
    ax.set_axis_off()

plt.tight_layout()

output_path = (
    FIGURE_DIR
    / "gistar_hotspot_coldspot_overlap_maps.png"
)

plt.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print("Saved:", output_path)

In [ ]:
# ============================================================
# Spatial OOF Gi* — Cell 1
# Restart-safe imports and reload validated OOF predictions
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from IPython.display import display

from libpysal.weights import Queen
from esda.getisord import G_Local
from statsmodels.stats.multitest import multipletests

RANDOM_STATE = 42


# ============================================================
# Project paths — same structure as current 004 notebook
# ============================================================

import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from project_config import RAW_DATA_DIR, WORK_DIR

PROJECT_DATA_DIR = WORK_DIR

OUTPUT_DIR = (
    PROJECT_DATA_DIR
    / "final_prediction_500m_resnet50_imagenet"
)

FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
SPATIAL_OUTPUT_DIR = OUTPUT_DIR / "spatial"

for directory in [
    FIGURE_DIR,
    TABLE_DIR,
    SPATIAL_OUTPUT_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# Preferred source:
# final GeoPackage already exported by current 004 notebook
# ============================================================

FINAL_OOF_GPKG = (
    SPATIAL_OUTPUT_DIR
    / "final_oof_predictions_500m.gpkg"
)

GRID_PATH = (
    PROJECT_DATA_DIR
    / "grid_size_selection"
    / "grids"
    / "grid_features_500m.gpkg"
)

OOF_PREDICTION_PATH = (
    PROJECT_DATA_DIR
    / "multimodal_fusion_500m"
    / "multimodal_oof_predictions.csv"
)


# ============================================================
# Load final validated predictions
# ============================================================

if FINAL_OOF_GPKG.exists():

    print(
        "Loading existing 004 final OOF GeoPackage..."
    )

    oof_gdf = gpd.read_file(
        FINAL_OOF_GPKG
    )

else:

    print(
        "Final 004 GeoPackage not found."
    )
    print(
        "Rebuilding it from the existing 003 OOF CSV "
        "without retraining any model..."
    )

    grid = gpd.read_file(
        GRID_PATH
    )[
        [
            "grid_id",
            "geometry",
        ]
    ].copy()

    oof_predictions = pd.read_csv(
        OOF_PREDICTION_PATH
    )

    # Final model = Tabular
    final_predictions = (
        oof_predictions[
            oof_predictions["feature_set"]
            == "Tabular"
        ]
        .copy()
    )

    # Keep raw target if target_mode exists
    if "target_mode" in final_predictions.columns:
        final_predictions = (
            final_predictions[
                final_predictions["target_mode"]
                == "raw"
            ]
            .copy()
        )

    random_oof = (
        final_predictions[
            final_predictions["validation"]
            == "random_5fold"
        ][
            [
                "grid_id",
                "observed",
                "predicted",
            ]
        ]
        .rename(
            columns={
                "predicted":
                    "random_predicted",
            }
        )
        .copy()
    )

    spatial_oof = (
        final_predictions[
            final_predictions["validation"]
            == "spatial_kmeans_leave_one_out"
        ][
            [
                "grid_id",
                "predicted",
            ]
        ]
        .rename(
            columns={
                "predicted":
                    "spatial_predicted",
            }
        )
        .copy()
    )

    oof_gdf = (
        grid
        .merge(
            random_oof,
            on="grid_id",
            how="inner",
            validate="one_to_one",
        )
        .merge(
            spatial_oof,
            on="grid_id",
            how="inner",
            validate="one_to_one",
        )
    )

    oof_gdf = gpd.GeoDataFrame(
        oof_gdf,
        geometry="geometry",
        crs=grid.crs,
    )


# ============================================================
# Checks
# ============================================================

required_columns = {
    "grid_id",
    "observed",
    "random_predicted",
    "spatial_predicted",
    "geometry",
}

missing_columns = (
    required_columns
    - set(oof_gdf.columns)
)

if missing_columns:
    raise KeyError(
        f"Missing required columns: "
        f"{sorted(missing_columns)}"
    )

if not oof_gdf["grid_id"].is_unique:
    raise ValueError(
        "grid_id is not unique."
    )

print(
    "\nRows loaded:",
    len(oof_gdf),
)

print(
    "Unique grids:",
    oof_gdf["grid_id"].nunique(),
)

print(
    "\nMissing values:"
)

display(
    oof_gdf[
        [
            "observed",
            "random_predicted",
            "spatial_predicted",
        ]
    ]
    .isna()
    .sum()
    .to_frame("missing")
)

oof_gdf.head()

In [ ]:
# ============================================================
# Spatial OOF Gi* — Cell 2
# Common Gi* sample, Queen weights and functions
# ============================================================


# ============================================================
# Use exactly the same grids for:
# observed, Random OOF, and Spatial OOF
# ============================================================

gi_gdf = (
    oof_gdf[
        [
            "grid_id",
            "observed",
            "random_predicted",
            "spatial_predicted",
            "geometry",
        ]
    ]
    .dropna(
        subset=[
            "observed",
            "random_predicted",
            "spatial_predicted",
        ]
    )
    .set_index("grid_id")
    .copy()
)


# ============================================================
# Initial Queen contiguity
# ============================================================

w_initial = Queen.from_dataframe(
    gi_gdf,
    use_index=True,
)

islands = list(
    w_initial.islands
)

print(
    "Initial Gi* grids:",
    len(gi_gdf),
)

print(
    "Queen-contiguity islands:",
    len(islands),
)


# ============================================================
# Exclude islands exactly once,
# so all three surfaces use the same valid sample
# ============================================================

if len(islands) > 0:

    gi_valid = (
        gi_gdf
        .drop(
            index=islands
        )
        .copy()
    )

else:

    gi_valid = (
        gi_gdf.copy()
    )


# Rebuild Queen weights
w = Queen.from_dataframe(
    gi_valid,
    use_index=True,
)

w.transform = "R"

print(
    "Final Gi* grids:",
    len(gi_valid),
)


# ============================================================
# Gi* calculation
# Same specification as original 004:
# - Queen contiguity
# - row-standardised weights
# - 999 permutations
# - Gi*
# - Benjamini-Hochberg FDR, alpha=0.05
# ============================================================

def calculate_gistar(
    values,
    weights,
    seed=42,
):

    gi_result = G_Local(
        np.asarray(values),
        weights,
        transform="R",
        permutations=999,
        star=True,
        seed=seed,
    )

    z_scores = (
        gi_result.Zs
    )

    p_raw = (
        gi_result.p_sim
    )

    # FDR correction
    p_fdr = multipletests(
        p_raw,
        alpha=0.05,
        method="fdr_bh",
    )[1]

    classification = np.full(
        len(values),
        "Not significant",
        dtype=object,
    )

    classification[
        (z_scores > 0)
        & (p_fdr < 0.05)
    ] = "Hot spot"

    classification[
        (z_scores < 0)
        & (p_fdr < 0.05)
    ] = "Cold spot"

    return (
        z_scores,
        p_raw,
        p_fdr,
        classification,
    )


# ============================================================
# Overlap metrics
# ============================================================

def calculate_overlap_metrics(
    observed_class,
    predicted_class,
    class_name,
):

    observed_mask = (
        observed_class
        == class_name
    )

    predicted_mask = (
        predicted_class
        == class_name
    )

    overlap_mask = (
        observed_mask
        & predicted_mask
    )

    union_mask = (
        observed_mask
        | predicted_mask
    )

    n_observed = int(
        observed_mask.sum()
    )

    n_predicted = int(
        predicted_mask.sum()
    )

    n_overlap = int(
        overlap_mask.sum()
    )

    n_union = int(
        union_mask.sum()
    )

    return {
        "Class":
            class_name,

        "Observed grids":
            n_observed,

        "Predicted grids":
            n_predicted,

        "Overlap grids":
            n_overlap,

        "Jaccard":
            (
                n_overlap / n_union
                if n_union > 0
                else np.nan
            ),

        "Observed recovery":
            (
                n_overlap / n_observed
                if n_observed > 0
                else np.nan
            ),

        "Predicted confirmation":
            (
                n_overlap / n_predicted
                if n_predicted > 0
                else np.nan
            ),
    }

In [ ]:
# ============================================================
# Spatial OOF Gi* — Cell 3
# Calculate Observed, Random OOF and Spatial OOF Gi*
# ============================================================


# ============================================================
# Observed Gi*
# ============================================================

(
    gi_valid["observed_GiZ"],
    gi_valid["observed_p_raw"],
    gi_valid["observed_p_fdr"],
    gi_valid["observed_gi_class"],
) = calculate_gistar(
    gi_valid[
        "observed"
    ].to_numpy(),
    w,
    seed=RANDOM_STATE,
)


# ============================================================
# Random OOF Gi*
# ============================================================

(
    gi_valid["random_GiZ"],
    gi_valid["random_p_raw"],
    gi_valid["random_p_fdr"],
    gi_valid["random_gi_class"],
) = calculate_gistar(
    gi_valid[
        "random_predicted"
    ].to_numpy(),
    w,
    seed=RANDOM_STATE,
)


# ============================================================
# Spatial OOF Gi*
# ============================================================

(
    gi_valid["spatial_GiZ"],
    gi_valid["spatial_p_raw"],
    gi_valid["spatial_p_fdr"],
    gi_valid["spatial_gi_class"],
) = calculate_gistar(
    gi_valid[
        "spatial_predicted"
    ].to_numpy(),
    w,
    seed=RANDOM_STATE,
)


# ============================================================
# Class counts
# ============================================================

print(
    "\nObserved Gi* classes:"
)
print(
    gi_valid[
        "observed_gi_class"
    ].value_counts()
)

print(
    "\nRandom OOF Gi* classes:"
)
print(
    gi_valid[
        "random_gi_class"
    ].value_counts()
)

print(
    "\nSpatial OOF Gi* classes:"
)
print(
    gi_valid[
        "spatial_gi_class"
    ].value_counts()
)


# ============================================================
# Compare Random and Spatial OOF against the SAME observed Gi*
# ============================================================

comparison_rows = []

for prediction_name, prediction_column in [
    (
        "Random OOF",
        "random_gi_class",
    ),
    (
        "Spatial OOF",
        "spatial_gi_class",
    ),
]:

    for class_name in [
        "Hot spot",
        "Cold spot",
    ]:

        row = calculate_overlap_metrics(
            gi_valid[
                "observed_gi_class"
            ],
            gi_valid[
                prediction_column
            ],
            class_name,
        )

        row[
            "Prediction"
        ] = prediction_name

        comparison_rows.append(
            row
        )


gi_comparison_table = pd.DataFrame(
    comparison_rows
)[
    [
        "Prediction",
        "Class",
        "Observed grids",
        "Predicted grids",
        "Overlap grids",
        "Jaccard",
        "Observed recovery",
        "Predicted confirmation",
    ]
]


print(
    "\nRandom vs Spatial OOF Gi* agreement:"
)

display(
    gi_comparison_table
    .round(4)
)


# ============================================================
# Direct change from Random -> Spatial
# ============================================================

gi_delta_table = (
    gi_comparison_table
    .pivot(
        index="Class",
        columns="Prediction",
        values=[
            "Jaccard",
            "Observed recovery",
            "Predicted confirmation",
        ],
    )
)

# Flatten column names
gi_delta_table.columns = [
    f"{metric}_{prediction}"
    for metric, prediction
    in gi_delta_table.columns
]

gi_delta_table = (
    gi_delta_table
    .reset_index()
)

for metric in [
    "Jaccard",
    "Observed recovery",
    "Predicted confirmation",
]:

    gi_delta_table[
        f"{metric}_Spatial_minus_Random"
    ] = (
        gi_delta_table[
            f"{metric}_Spatial OOF"
        ]
        - gi_delta_table[
            f"{metric}_Random OOF"
        ]
    )


print(
    "\nChange from Random OOF to Spatial OOF:"
)

display(
    gi_delta_table
    .round(4)
)


# ============================================================
# Save tables
# ============================================================

comparison_path = (
    TABLE_DIR
    / "gistar_random_vs_spatial_oof_overlap.csv"
)

delta_path = (
    TABLE_DIR
    / "gistar_random_vs_spatial_oof_change.csv"
)

gi_comparison_table.to_csv(
    comparison_path,
    index=False,
)

gi_delta_table.to_csv(
    delta_path,
    index=False,
)

print(
    "\nSaved:",
    comparison_path,
)

print(
    "Saved:",
    delta_path,
)

In [ ]:
# ============================================================
# Spatial OOF Gi* — Cell 4
# Observed vs Random OOF vs Spatial OOF Gi* maps
# ============================================================

from matplotlib.patches import Patch


gi_class_colors = {
    "Cold spot":
        "#2166ac",
    "Not significant":
        "#eeeeee",
    "Hot spot":
        "#b2182b",
}


fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 7),
)


plot_specs = [
    (
        "observed_gi_class",
        "Observed electricity Gi*",
    ),
    (
        "random_gi_class",
        "Random OOF Gi*",
    ),
    (
        "spatial_gi_class",
        "Spatial OOF Gi*",
    ),
]


for ax, (
    column,
    title,
) in zip(
    axes,
    plot_specs,
):

    # Base layer
    gi_valid.plot(
        ax=ax,
        color="white",
        edgecolor="#d9d9d9",
        linewidth=0.2,
    )

    for category in [
        "Cold spot",
        "Not significant",
        "Hot spot",
    ]:

        subset = (
            gi_valid[
                gi_valid[column]
                == category
            ]
        )

        if not subset.empty:

            subset.plot(
                ax=ax,
                color=(
                    gi_class_colors[
                        category
                    ]
                ),
                edgecolor="none",
            )

    ax.set_title(
        title,
        fontsize=12,
    )

    ax.set_axis_off()


# ============================================================
# Shared legend
# ============================================================

legend_handles = [
    Patch(
        facecolor=(
            gi_class_colors[
                "Hot spot"
            ]
        ),
        edgecolor="none",
        label="Hot spot",
    ),
    Patch(
        facecolor=(
            gi_class_colors[
                "Cold spot"
            ]
        ),
        edgecolor="none",
        label="Cold spot",
    ),
    Patch(
        facecolor=(
            gi_class_colors[
                "Not significant"
            ]
        ),
        edgecolor="none",
        label="Not significant",
    ),
]


axes[2].legend(
    handles=legend_handles,
    title="Gi* class",
    loc="upper right",
    frameon=True,
)


plt.tight_layout()


output_path = (
    FIGURE_DIR
    / "observed_random_spatial_gistar_maps.png"
)

plt.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print(
    "Saved:",
    output_path,
)

In [ ]:
# ============================================================
# Spatial OOF Gi* — Cell 5
# Random vs Spatial agreement with observed Gi*
# ============================================================

from matplotlib.patches import Patch


# ============================================================
# Agreement classification function
# ============================================================

def build_agreement_class(
    observed_class,
    predicted_class,
):

    agreement = np.full(
        len(observed_class),
        "Other / mismatch",
        dtype=object,
    )

    hot_overlap = (
        (observed_class == "Hot spot")
        & (predicted_class == "Hot spot")
    )

    cold_overlap = (
        (observed_class == "Cold spot")
        & (predicted_class == "Cold spot")
    )

    both_not_significant = (
        (observed_class == "Not significant")
        & (predicted_class == "Not significant")
    )

    agreement[
        hot_overlap
    ] = "Hot spot overlap"

    agreement[
        cold_overlap
    ] = "Cold spot overlap"

    agreement[
        both_not_significant
    ] = "Both not significant"

    return agreement


gi_valid[
    "random_gi_agreement"
] = build_agreement_class(
    gi_valid[
        "observed_gi_class"
    ].to_numpy(),
    gi_valid[
        "random_gi_class"
    ].to_numpy(),
)

gi_valid[
    "spatial_gi_agreement"
] = build_agreement_class(
    gi_valid[
        "observed_gi_class"
    ].to_numpy(),
    gi_valid[
        "spatial_gi_class"
    ].to_numpy(),
)


# ============================================================
# Colours
# ============================================================

agreement_colors = {
    "Hot spot overlap":
        "#b2182b",
    "Cold spot overlap":
        "#2166ac",
    "Both not significant":
        "#eeeeee",
    "Other / mismatch":
        "#f4a261",
}


# ============================================================
# Plot Random and Spatial agreement side-by-side
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(15, 7),
)

plot_specs = [
    (
        "random_gi_agreement",
        "Observed vs Random OOF Gi* Agreement",
    ),
    (
        "spatial_gi_agreement",
        "Observed vs Spatial OOF Gi* Agreement",
    ),
]

plot_order = [
    "Both not significant",
    "Other / mismatch",
    "Cold spot overlap",
    "Hot spot overlap",
]


for ax, (
    column,
    title,
) in zip(
    axes,
    plot_specs,
):

    for category in plot_order:

        subset = (
            gi_valid[
                gi_valid[column]
                == category
            ]
        )

        if not subset.empty:

            subset.plot(
                ax=ax,
                color=(
                    agreement_colors[
                        category
                    ]
                ),
                edgecolor="none",
            )

    ax.set_title(
        title,
        fontsize=12,
    )

    ax.set_axis_off()


# ============================================================
# Shared legend
# ============================================================

legend_handles = [
    Patch(
        facecolor=(
            agreement_colors[
                "Hot spot overlap"
            ]
        ),
        edgecolor="none",
        label="Hot spot overlap",
    ),
    Patch(
        facecolor=(
            agreement_colors[
                "Cold spot overlap"
            ]
        ),
        edgecolor="none",
        label="Cold spot overlap",
    ),
    Patch(
        facecolor=(
            agreement_colors[
                "Both not significant"
            ]
        ),
        edgecolor="none",
        label="Both not significant",
    ),
    Patch(
        facecolor=(
            agreement_colors[
                "Other / mismatch"
            ]
        ),
        edgecolor="none",
        label="Other / mismatch",
    ),
]


axes[1].legend(
    handles=legend_handles,
    title="Agreement class",
    loc="upper right",
    frameon=True,
)


plt.tight_layout()


agreement_path = (
    FIGURE_DIR
    / "gistar_random_vs_spatial_agreement_maps.png"
)

plt.savefig(
    agreement_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print(
    "Saved:",
    agreement_path,
)


# ============================================================
# Export full Gi* spatial layer
# ============================================================

gi_export_path = (
    SPATIAL_OUTPUT_DIR
    / "gistar_random_spatial_oof_500m.gpkg"
)

gi_valid.reset_index().to_file(
    gi_export_path,
    driver="GPKG",
)

print(
    "Saved Gi* GeoPackage:",
    gi_export_path,
)

In [ ]:
# ============================================================
# Spatial comparison with urban morphology
# ============================================================

fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 7),
)

plot_specs = [
    (
        "random_predicted",
        "Predicted electricity consumption",
        "Reds",
    ),
    (
        "building_coverage",
        "Building coverage",
        "Blues",
    ),
    (
        "road_density",
        "Road density",
        "Greys",
    ),
]

for ax, (
    column,
    title,
    cmap,
) in zip(
    axes,
    plot_specs,
):

    values = analysis_gdf[
        column
    ].dropna()

    vmin = values.quantile(0.02)
    vmax = values.quantile(0.98)

    analysis_gdf.plot(
        column=column,
        ax=ax,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        edgecolor="none",
        legend=True,
        legend_kwds={
            "shrink": 0.7,
        },
    )

    ax.set_title(title)
    ax.set_axis_off()

fig.suptitle(
    "Predicted Electricity Consumption and Urban Morphology",
    fontsize=15,
)

plt.tight_layout()

output_path = (
    FIGURE_DIR
    / "electricity_vs_urban_morphology_maps.png"
)

plt.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
# ============================================================
# Build observed and predicted low-consumption/high-density proxy
# ============================================================

proxy_gdf = analysis_gdf.copy()

# Use the same valid grids for observed and predicted comparisons
valid = (
    proxy_gdf["observed"].notna()
    & proxy_gdf["random_predicted"].notna()
    & proxy_gdf["pop_density_km2"].notna()
)

# Thresholds are defined only from observed data
population_threshold = proxy_gdf.loc[
    valid,
    "pop_density_km2",
].quantile(0.75)

electricity_threshold = proxy_gdf.loc[
    valid,
    "observed",
].quantile(0.25)

# Observed low-consumption / high-density proxy
proxy_gdf["observed_proxy"] = (
    valid
    & (
        proxy_gdf["pop_density_km2"]
        >= population_threshold
    )
    & (
        proxy_gdf["observed"]
        <= electricity_threshold
    )
)

# Predicted proxy uses exactly the same thresholds
proxy_gdf["predicted_proxy"] = (
    valid
    & (
        proxy_gdf["pop_density_km2"]
        >= population_threshold
    )
    & (
        proxy_gdf["random_predicted"]
        <= electricity_threshold
    )
)

print(
    "Population density threshold:",
    round(population_threshold, 2),
)

print(
    "Observed electricity threshold:",
    round(electricity_threshold, 2),
)

print(
    "Observed proxy grids:",
    int(proxy_gdf["observed_proxy"].sum()),
)

print(
    "Predicted proxy grids:",
    int(proxy_gdf["predicted_proxy"].sum()),
)

In [ ]:
# ============================================================
# Observed vs predicted proxy maps
# ============================================================

pop_vmin = proxy_gdf[
    "pop_density_km2"
].quantile(0.02)

pop_vmax = proxy_gdf[
    "pop_density_km2"
].quantile(0.98)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 7),
)

proxy_specs = [
    (
        "observed_proxy",
        "Observed low-consumption / high-density proxy",
    ),
    (
        "predicted_proxy",
        "Predicted low-consumption / high-density proxy",
    ),
]

for ax, (
    proxy_column,
    title,
) in zip(
    axes,
    proxy_specs,
):

    # Population-density background
    proxy_gdf.plot(
        column="pop_density_km2",
        ax=ax,
        cmap="Greys",
        vmin=pop_vmin,
        vmax=pop_vmax,
        edgecolor="none",
    )

    # Semi-transparent proxy areas
    proxy_gdf.loc[
        proxy_gdf[proxy_column]
    ].plot(
        ax=ax,
        color="#e75480",
        alpha=0.55,
        edgecolor="none",
    )

    ax.set_title(title)
    ax.set_axis_off()

# Shared population-density colourbar
sm = ScalarMappable(
    norm=Normalize(
        vmin=pop_vmin,
        vmax=pop_vmax,
    ),
    cmap="Greys",
)

cax = fig.add_axes(
    [0.92, 0.18, 0.015, 0.64]
)

cbar = fig.colorbar(
    sm,
    cax=cax,
)

cbar.set_label(
    "Population density (people/km²)"
)

plt.subplots_adjust(
    right=0.90,
    wspace=0.05,
)

output_path = (
    FIGURE_DIR
    / "observed_vs_predicted_low_consumption_high_density_proxy.png"
)

plt.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
# ============================================================
# Quantify overlap between observed and predicted proxy areas
# ============================================================

observed_mask = proxy_gdf[
    "observed_proxy"
]

predicted_mask = proxy_gdf[
    "predicted_proxy"
]

intersection = (
    observed_mask
    & predicted_mask
).sum()

union = (
    observed_mask
    | predicted_mask
).sum()

observed_total = (
    observed_mask.sum()
)

predicted_total = (
    predicted_mask.sum()
)

jaccard = (
    intersection / union
    if union > 0
    else np.nan
)

observed_recall = (
    intersection / observed_total
    if observed_total > 0
    else np.nan
)

predicted_precision = (
    intersection / predicted_total
    if predicted_total > 0
    else np.nan
)

comparison_table = pd.DataFrame(
    {
        "Metric": [
            "Observed proxy grids",
            "Predicted proxy grids",
            "Overlapping grids",
            "Jaccard overlap",
            "Observed areas recovered",
            "Predicted areas confirmed",
        ],
        "Value": [
            observed_total,
            predicted_total,
            intersection,
            jaccard,
            observed_recall,
            predicted_precision,
        ],
    }
)

display(comparison_table)

In [ ]:
# ============================================================
# Spatial agreement map with legend outside
# ============================================================

from matplotlib.patches import Patch

proxy_gdf["agreement"] = "Neither"

proxy_gdf.loc[
    proxy_gdf["observed_proxy"]
    & ~proxy_gdf["predicted_proxy"],
    "agreement",
] = "Observed only"

proxy_gdf.loc[
    ~proxy_gdf["observed_proxy"]
    & proxy_gdf["predicted_proxy"],
    "agreement",
] = "Predicted only"

proxy_gdf.loc[
    proxy_gdf["observed_proxy"]
    & proxy_gdf["predicted_proxy"],
    "agreement",
] = "Observed + predicted"

agreement_colors = {
    "Neither": "#eeeeee",
    "Observed only": "#4575b4",
    "Predicted only": "#fdae61",
    "Observed + predicted": "#d73027",
}

fig, ax = plt.subplots(
    figsize=(10, 8)
)

for category, color in agreement_colors.items():

    subset = proxy_gdf[
        proxy_gdf["agreement"] == category
    ]

    subset.plot(
        ax=ax,
        color=color,
        edgecolor="none",
    )

legend_handles = [
    Patch(
        facecolor="#eeeeee",
        edgecolor="none",
        label="Neither identified",
    ),
    Patch(
        facecolor="#4575b4",
        edgecolor="none",
        label="Observed only",
    ),
    Patch(
        facecolor="#fdae61",
        edgecolor="none",
        label="Predicted only",
    ),
    Patch(
        facecolor="#d73027",
        edgecolor="none",
        label="Observed + predicted",
    ),
]

ax.legend(
    handles=legend_handles,
    title="Proxy-area agreement",
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=True,
)

ax.set_title(
    "Agreement between Observed and Predicted Proxy Areas"
)

ax.set_axis_off()

plt.subplots_adjust(
    right=0.78
)

output_path = (
    FIGURE_DIR
    / "observed_predicted_proxy_overlap.png"
)

plt.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
# ============================================================
# High-building-density / low-consumption proxy
# ============================================================

building_proxy_gdf = analysis_gdf.copy()

# Use the same valid grids for observed and predicted comparisons
valid_building_proxy = (
    building_proxy_gdf["observed"].notna()
    & building_proxy_gdf["random_predicted"].notna()
    & building_proxy_gdf["building_coverage"].notna()
)

# Thresholds are derived from observed-data grids only
building_threshold = (
    building_proxy_gdf.loc[
        valid_building_proxy,
        "building_coverage",
    ]
    .quantile(0.75)
)

electricity_threshold = (
    building_proxy_gdf.loc[
        valid_building_proxy,
        "observed",
    ]
    .quantile(0.25)
)

# Observed proxy
building_proxy_gdf["observed_building_proxy"] = (
    valid_building_proxy
    & (
        building_proxy_gdf["building_coverage"]
        >= building_threshold
    )
    & (
        building_proxy_gdf["observed"]
        <= electricity_threshold
    )
)

# Predicted proxy
# Use exactly the same building and electricity thresholds
building_proxy_gdf["predicted_building_proxy"] = (
    valid_building_proxy
    & (
        building_proxy_gdf["building_coverage"]
        >= building_threshold
    )
    & (
        building_proxy_gdf["random_predicted"]
        <= electricity_threshold
    )
)

# ============================================================
# Overlap metrics
# ============================================================

observed_mask = (
    building_proxy_gdf["observed_building_proxy"]
)

predicted_mask = (
    building_proxy_gdf["predicted_building_proxy"]
)

overlap_mask = (
    observed_mask
    & predicted_mask
)

union_mask = (
    observed_mask
    | predicted_mask
)

n_observed = int(observed_mask.sum())
n_predicted = int(predicted_mask.sum())
n_overlap = int(overlap_mask.sum())
n_union = int(union_mask.sum())

jaccard = (
    n_overlap / n_union
    if n_union > 0
    else np.nan
)

observed_recovery = (
    n_overlap / n_observed
    if n_observed > 0
    else np.nan
)

predicted_confirmation = (
    n_overlap / n_predicted
    if n_predicted > 0
    else np.nan
)

building_proxy_metrics = pd.DataFrame({
    "Metric": [
        "Building coverage threshold (Q75)",
        "Electricity threshold (Q25)",
        "Observed proxy grids",
        "Predicted proxy grids",
        "Overlapping grids",
        "Jaccard overlap",
        "Observed areas recovered",
        "Predicted areas confirmed",
    ],
    "Value": [
        building_threshold,
        electricity_threshold,
        n_observed,
        n_predicted,
        n_overlap,
        jaccard,
        observed_recovery,
        predicted_confirmation,
    ],
})

display(
    building_proxy_metrics.round(4)
)

building_proxy_metrics.to_csv(
    TABLE_DIR
    / "high_building_low_consumption_proxy_metrics.csv",
    index=False,
)


# ============================================================
# Observed vs predicted proxy maps
# ============================================================

building_vmin = (
    building_proxy_gdf[
        "building_coverage"
    ]
    .quantile(0.02)
)

building_vmax = (
    building_proxy_gdf[
        "building_coverage"
    ]
    .quantile(0.98)
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 7),
)

proxy_specs = [
    (
        "observed_building_proxy",
        "Observed high-building-density / low-consumption proxy",
    ),
    (
        "predicted_building_proxy",
        "Predicted high-building-density / low-consumption proxy",
    ),
]

for ax, (
    proxy_column,
    title,
) in zip(
    axes,
    proxy_specs,
):

    building_proxy_gdf.plot(
        column="building_coverage",
        ax=ax,
        cmap="Greys",
        vmin=building_vmin,
        vmax=building_vmax,
        edgecolor="none",
    )

    building_proxy_gdf.loc[
        building_proxy_gdf[proxy_column]
    ].plot(
        ax=ax,
        color="#e75480",
        alpha=0.60,
        edgecolor="none",
    )

    ax.set_title(title)
    ax.set_axis_off()

sm = ScalarMappable(
    norm=Normalize(
        vmin=building_vmin,
        vmax=building_vmax,
    ),
    cmap="Greys",
)

cax = fig.add_axes(
    [0.92, 0.18, 0.015, 0.64]
)

cbar = fig.colorbar(
    sm,
    cax=cax,
)

cbar.set_label(
    "Building coverage"
)

plt.subplots_adjust(
    right=0.90,
    wspace=0.05,
)

output_path = (
    FIGURE_DIR
    / "observed_vs_predicted_high_building_low_consumption_proxy.png"
)

plt.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print("Saved:", output_path)

## 7. Export final spatial layers

The GeoPackage contains both Random-CV and Spatial-CV OOF predictions. These are the validated prediction layers to use for dissertation mapping and GIS inspection.

In [ ]:
# ============================================================
# Export final validated prediction layers
# ============================================================

random_export = (
    random_map_gdf[
        [
            "grid_id",
            "fold",
            "observed",
            "predicted",
            "residual",
            "absolute_error",
        ]
    ]
    .rename(
        columns={
            "fold": (
                "random_fold"
            ),
            "predicted": (
                "random_predicted"
            ),
            "residual": (
                "random_residual"
            ),
            "absolute_error": (
                "random_abs_error"
            ),
        }
    )
)

spatial_export = (
    spatial_map_gdf[
        [
            "grid_id",
            "fold",
            "predicted",
            "residual",
            "absolute_error",
        ]
    ]
    .rename(
        columns={
            "fold": (
                "spatial_fold"
            ),
            "predicted": (
                "spatial_predicted"
            ),
            "residual": (
                "spatial_residual"
            ),
            "absolute_error": (
                "spatial_abs_error"
            ),
        }
    )
)

final_spatial_gdf = (
    grid[
        [
            "grid_id",
            "geometry",
        ]
    ]
    .merge(
        random_export,
        on="grid_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        spatial_export,
        on="grid_id",
        how="left",
        validate="one_to_one",
    )
)

final_spatial_gdf = (
    gpd.GeoDataFrame(
        final_spatial_gdf,
        geometry="geometry",
        crs=grid.crs,
    )
)

final_gpkg_path = (
    SPATIAL_OUTPUT_DIR
    / "final_oof_predictions_500m.gpkg"
)

final_csv_path = (
    TABLE_DIR
    / "final_oof_predictions_500m.csv"
)

final_spatial_gdf.to_file(
    final_gpkg_path,
    driver="GPKG",
)

final_spatial_gdf.drop(
    columns="geometry"
).to_csv(
    final_csv_path,
    index=False,
)

print(
    "Saved GeoPackage:",
    final_gpkg_path,
)

print(
    "Saved CSV:",
    final_csv_path,
)

In [ ]:
# ============================================================
# Final output inventory
# ============================================================

print("004 output directory:")
print(OUTPUT_DIR)

print("\nGenerated figures:")
for path in sorted(
    FIGURE_DIR.glob("*")
):
    print(" -", path)

print("\nGenerated tables:")
for path in sorted(
    TABLE_DIR.glob("*")
):
    print(" -", path)

print("\nGenerated spatial files:")
for path in sorted(
    SPATIAL_OUTPUT_DIR.glob("*")
):
    print(" -", path)